The following notebook runs test searches on specified files in the dataset (defined in input_files). Each sentence in the document is searched for using its key words. The sentence's ranking in the search that used its own key words is saved. All results are saved in a `.jsonl` with the naming convention `data/win_<window_offset>-q_<search_lang>-a_<source_lang>-topk_<rerank_top_k>.jsonl'`

In [ ]:
from datetime import datetime, timezone
import gc
import json
import math
import pickle
from pprint import pprint
import random

import pandas as pd
import spacy
import torch
from transformers import AutoModel
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from yasbd import register_spacy_component

from app.searcher import Searcher

In [ ]:
with open('data/sentence_dataset.pkl', 'rb') as file_path:
    print('Loading dataset...')

    dataset = pickle.load(file_path)
    print('Dataset loaded successfully.\n\nExample sentence pair: ')
    doc = next(iter(dataset.items()))
    pprint(doc[1][0])

In [ ]:
dataset['ep-97-03-11'][2]

In [ ]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

tokenizer = AutoTokenizer.from_pretrained(
    'jinaai/jina-embeddings-v2-base-es',
    trust_remote_code=True,
    local_files_only=True,
    max_length=8192
)

model = AutoModel.from_pretrained(
    'jinaai/jina-embeddings-v2-base-es',
    trust_remote_code=True,
    local_files_only=True
).to(device)

reranker = AutoModelForSequenceClassification.from_pretrained(
    'jinaai/jina-reranker-v2-base-multilingual',
    dtype="auto",
    trust_remote_code=True,
).to(device)

register_spacy_component()
spacy_model = spacy.blank('xx')
spacy_model.add_pipe('yasbd', first=True, config={'lang': 'auto'})

In [ ]:
window_offset = 1
search_lang = 'es'
source_lang = 'es'
rerank_top_k = 0
limit_1 = False # Set False to run on all input_files at once

input_files = {
    'ep-97-03-11',
    'ep-99-03-23',
    'ep-99-02-11',
    'ep-01-04-02',
    'ep-10-02-09-004',
    'ep-10-02-10-010',
    'ep-11-06-07-009',
    'ep-03-06-04',
    'ep-01-01-18',
    'ep-07-05-23-009',
    'pan-22-5',
    'pan-62-12',
    'pan-46-12',
    'pan-140-2',
    'pan-12-4',
    'pan-238-24',
    'pan-35-33',
    'pan-35-20',
    'pan-107-11',
    'pan-136-0'
}

In [ ]:
log_path = f'data/win_{window_offset}-q_{search_lang}-a_{source_lang}-topk_{rerank_top_k}.jsonl'

def log_search(search_res, log_file=log_path):
    search_res['timestamp'] = datetime.now(timezone.utc).isoformat()
    with open(log_file, 'a', encoding='utf-8') as log_file:
        log_file.write(json.dumps(search_res) + '\n')

In [ ]:
# If resuming an evaluation, identify remaining unevaluated files
try:
    with open(log_path, 'r', encoding='utf-8') as f:
        searched_files = {json.loads(line)['file_name'] for line in f if line.strip()}
        input_files = input_files - searched_files
except FileNotFoundError:
    pass

# Breaks after the 1st iteration (recommended)
for file_name in input_files:
    # Construct input_text from all the sentences in each doc
    input_text = ' '.join([sent_obj[source_lang] for sent_obj in dataset[file_name]][:100])

    searcher = Searcher(
        input_text=input_text,
        nlp=spacy_model,
        tokenizer=tokenizer,
        model=model,
        max_chunk_size=8192,
        reranker=reranker,
        window_offset=window_offset
    )

    rng = random.Random(42)
    for sent_obj in dataset[file_name]:
        sentence = sent_obj[source_lang]
        sentence_bounds = sent_obj[f'{source_lang}_bounds']
        phrases = sent_obj[f'{search_lang}_phrases']

        search_query = rng.choice(phrases)

        search_results = searcher.get_ranked_similarities(search_query, rerank_top_k=5)

        for i, search_result in enumerate(search_results):
            if (sentence_bounds[0] < search_result.chunk.char_bounds[1] and
                search_result.chunk.char_bounds[0] < sentence_bounds[1]):

                search_res = {
                    'query': search_query,
                    'target': sentence,
                    'file_name': file_name,
                    'char_start': sentence_bounds[0],
                    'char_end': sentence_bounds[1],
                    'rank': i + 1,
                    'all_results': [result.chunk.char_bounds for result in search_results]
                }
                log_search(search_res)
                break
            elif i == len(search_results) - 1:
                print(sentence)
                print('source', sentence_bounds)
                print('result', search_result.chunk.char_bounds)

    # Clear up memory
    del searcher
    gc.collect()

    if limit_1:
        break
